# Wine Classification Quickstart

Notebook singkat ini membuat baseline klasifikasi wine menggunakan dataset bawaan scikit-learn. Fokusnya adalah alur yang cepat: load data, eksplorasi ringkas, training model, evaluasi, dan insight feature importance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, accuracy_score
from sklearn.model_selection import train_test_split

plt.style.use("seaborn-v0_8-whitegrid")

## Load Dataset

Dataset Wine berisi 178 sampel dengan 13 fitur numerik. Targetnya adalah 3 kelas wine.

In [ ]:
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df["target"] = wine.target
df["class_name"] = df["target"].map(dict(enumerate(wine.target_names)))

print(df.shape)
df.head()

In [ ]:
df.describe().T

## Distribusi Kelas

Kita cek apakah data terlalu berat ke salah satu kelas.

In [ ]:
class_counts = df["class_name"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
class_counts.plot(kind="bar", ax=ax, color=["#2f80ed", "#27ae60", "#f2994a"])
ax.set_title("Jumlah Sampel per Kelas")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
plt.xticks(rotation=0)
plt.show()

class_counts

## Korelasi Fitur

Heatmap sederhana membantu melihat fitur mana yang cenderung bergerak bersama.

In [ ]:
corr = df[wine.feature_names].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticklabels(corr.columns)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Feature Correlation")
plt.tight_layout()
plt.show()

## Train Baseline Model

Random Forest cocok untuk baseline cepat karena kuat untuk fitur tabular dan memberi feature importance.

In [ ]:
X = df[wine.feature_names]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
accuracy = accuracy_score(y_test, preds)
print(f"Accuracy: {accuracy:.3f}")

In [ ]:
print(classification_report(y_test, preds, target_names=wine.target_names))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    preds,
    display_labels=wine.target_names,
    cmap="Blues",
)
plt.title("Confusion Matrix")
plt.show()

## Feature Importance

Fitur dengan importance tertinggi adalah kandidat utama untuk dianalisis lebih lanjut.

In [ ]:
importance = pd.Series(model.feature_importances_, index=wine.feature_names).sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
importance.tail(10).plot(kind="barh", ax=ax, color="#2f80ed")
ax.set_title("Top 10 Feature Importance")
ax.set_xlabel("Importance")
plt.show()

importance.sort_values(ascending=False).head(10).to_frame("importance")

## Kesimpulan

Baseline Random Forest sudah cukup kuat untuk dataset kecil ini. Langkah lanjutan yang menarik adalah membandingkan model lain seperti Logistic Regression atau SVM, lalu menggunakan cross-validation agar evaluasinya lebih stabil.